# locations_mapped — nightly rebuild (oxjob #762 / #765)

`locations_mapped` is a nightly `CREATE OR REPLACE`: `locations_w_types` JOIN the
`location_work_ids` identity registry (maintained by the `Map_Work_Ids` task,
oxjob #764), UNION the frozen `locations_stale` sidecar. `location_enrichments`
overlays dormant-writer columns works_base still consumes. `locations_mapped_hash`
recreates the MERGE-era `openalex_updated_dt` semantics: stamps carry when content
is unchanged, bump when it is not. Design + evidence: oxjobs #762 PLAN-v2, #765.

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.location_enrichments') (
  provenance STRING,
  native_id_namespace STRING,
  native_id STRING,
  language_classification STRUCT<language: STRING, score: DOUBLE>,
  referenced_works_count INT,
  referenced_works ARRAY<BIGINT>,
  snapshot_dt DATE
)
CLUSTER BY (provenance, native_id_namespace, native_id)

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.locations_stale') (
  work_id BIGINT,
  work_id_source STRING,
  merge_key STRUCT<doi: STRING, pmid: STRING, arxiv: STRING, title_author: STRING>,
  key_lineage STRING,
  provenance STRING,
  native_id STRING,
  native_id_namespace STRING,
  title STRING,
  normalized_title STRING,
  authors ARRAY<STRUCT<given: STRING, family: STRING, name: STRING, orcid: STRING, affiliations: ARRAY<STRUCT<name: STRING, department: STRING, ror_id: STRING>>, is_corresponding: BOOLEAN, author_key: STRING>>,
  ids ARRAY<STRUCT<id: STRING, namespace: STRING, relationship: STRING>>,
  raw_type STRING,
  type STRING,
  version STRING,
  license STRING,
  language STRING,
  language_classification STRUCT<language: STRING, score: DOUBLE>,
  published_date DATE,
  created_date DATE,
  updated_date DATE,
  issue STRING,
  volume STRING,
  first_page STRING,
  last_page STRING,
  is_retracted BOOLEAN,
  abstract STRING,
  source_name STRING,
  publisher STRING,
  funders ARRAY<STRUCT<doi: STRING, ror: STRING, name: STRING, awards: ARRAY<STRING>>>,
  references ARRAY<STRUCT<doi: STRING, pmid: STRING, arxiv: STRING, title: STRING, authors: STRING, year: STRING, raw: STRING>>,
  urls ARRAY<STRUCT<url: STRING, content_type: STRING>>,
  pdf_url STRING,
  landing_page_url STRING,
  pdf_s3_id STRING,
  grobid_s3_id STRING,
  mesh STRING,
  is_oa BOOLEAN,
  is_oa_source BOOLEAN,
  referenced_works_count INT,
  referenced_works ARRAY<BIGINT>,
  abstract_inverted_index STRING,
  authors_exist BOOLEAN,
  affiliations_exist BOOLEAN,
  is_corresponding_exists BOOLEAN,
  best_doi STRING,
  source_id BIGINT,
  openalex_created_dt DATE,
  openalex_updated_dt TIMESTAMP,
  violates_landing_rule BOOLEAN,
  snapshot_dt DATE,
  stale_reason STRING
)

## Hash baseline (first run only) — change detection for `openalex_updated_dt`
Payload excludes the two bookkeeping stamps. Formula changes require a rebaseline
from pre-change rows or the next run bumps everything (#733 08-07 incident).

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.locations_mapped_hash')
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
SELECT provenance, native_id_namespace, native_id, work_id,
       xxhash64(to_json(struct(
         work_id, work_id_source, merge_key, key_lineage, title, normalized_title,
         authors, ids, raw_type, type, version, license, language, language_classification,
         published_date, created_date, updated_date, issue, volume, first_page, last_page,
         is_retracted, abstract, source_name, publisher, funders, references, urls,
         pdf_url, landing_page_url, pdf_s3_id, grobid_s3_id, mesh, is_oa, is_oa_source,
         referenced_works_count, referenced_works, abstract_inverted_index,
         authors_exist, affiliations_exist, is_corresponding_exists, best_doi, source_id
       ))) AS payload_hash,
       openalex_updated_dt
FROM identifier('openalex' || :env_suffix || '.works.locations_mapped')
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY provenance, native_id_namespace, native_id, work_id
  ORDER BY openalex_updated_dt DESC NULLS LAST) = 1

## Rebuild `locations_mapped`
Live side: `locations_w_types` (one row per anchor) JOIN registry JOIN enrichment
overlay. Stale side: sidecar rows not already represented live — the anti-join is at
`(anchor, work_id)` granularity so preserved twin rows (`stale_reason = 'twin_alive'`)
survive alongside their live sibling; a returning record with the same work_id is
not duplicated. Stamps carry over when the payload hash is unchanged.

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.locations_mapped')
CLUSTER BY (merge_key.doi, merge_key.pmid, merge_key.arxiv, merge_key.title_author)
TBLPROPERTIES (
  'delta.checkpoint.writeStatsAsJson' = 'false',
  'delta.checkpoint.writeStatsAsStruct' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.feature.deletionVectors' = 'supported',
  'delta.feature.rowTracking' = 'supported',
  'delta.feature.v2Checkpoint' = 'supported')
AS
WITH t AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY provenance, native_id_namespace, native_id
           ORDER BY updated_date DESC) AS rwcnt
  FROM identifier('openalex' || :env_suffix || '.works.locations_w_types')
  QUALIFY rwcnt = 1
),
live AS (
  SELECT
    r.work_id,
    r.work_id_source,
    t.merge_key,
    CAST(NULL AS STRING) AS key_lineage,
    t.provenance,
    t.native_id,
    t.native_id_namespace,
    t.title,
    t.normalized_title,
    t.authors,
    t.ids,
    t.raw_type,
    t.type,
    t.version,
    t.license,
    t.language,
    e.language_classification,
    t.published_date,
    t.created_date,
    t.updated_date,
    t.issue,
    t.volume,
    t.first_page,
    t.last_page,
    COALESCE(t.is_retracted, FALSE) AS is_retracted,
    t.abstract,
    t.source_name,
    t.publisher,
    t.funders,
    t.references,
    t.urls,
    t.pdf_url,
    t.landing_page_url,
    t.pdf_s3_id,
    t.grobid_s3_id,
    t.mesh,
    COALESCE(t.is_oa, FALSE) AS is_oa,
    COALESCE(t.is_oa_source, FALSE) AS is_oa_source,
    e.referenced_works_count,
    e.referenced_works,
    t.abstract_inverted_index,
    t.authors_exist,
    t.affiliations_exist,
    t.is_corresponding_exists,
    t.best_doi,
    t.source_id,
    COALESCE(r.openalex_created_dt, current_date()) AS openalex_created_dt
  FROM t
  LEFT JOIN identifier('openalex' || :env_suffix || '.works.location_work_ids') r
    ON  t.provenance = r.provenance
    AND t.native_id_namespace = r.native_id_namespace
    AND t.native_id = r.native_id
  LEFT JOIN identifier('openalex' || :env_suffix || '.works.location_enrichments') e
    ON  t.provenance = e.provenance
    AND t.native_id_namespace = e.native_id_namespace
    AND t.native_id = e.native_id
),
stale AS (
  SELECT
    s.work_id,
    s.work_id_source,
    s.merge_key,
    s.key_lineage,
    s.provenance,
    s.native_id,
    s.native_id_namespace,
    s.title,
    s.normalized_title,
    s.authors,
    s.ids,
    s.raw_type,
    s.type,
    s.version,
    s.license,
    s.language,
    s.language_classification,
    s.published_date,
    s.created_date,
    s.updated_date,
    s.issue,
    s.volume,
    s.first_page,
    s.last_page,
    s.is_retracted,
    s.abstract,
    s.source_name,
    s.publisher,
    s.funders,
    s.references,
    s.urls,
    s.pdf_url,
    s.landing_page_url,
    s.pdf_s3_id,
    s.grobid_s3_id,
    s.mesh,
    s.is_oa,
    s.is_oa_source,
    s.referenced_works_count,
    s.referenced_works,
    s.abstract_inverted_index,
    s.authors_exist,
    s.affiliations_exist,
    s.is_corresponding_exists,
    s.best_doi,
    s.source_id,
    s.openalex_created_dt
  FROM identifier('openalex' || :env_suffix || '.works.locations_stale') s
  LEFT ANTI JOIN live l
    ON  s.provenance = l.provenance
    AND s.native_id_namespace = l.native_id_namespace
    AND s.native_id = l.native_id
    AND s.work_id <=> l.work_id
),
unioned AS (
  SELECT * FROM live
  UNION ALL
  SELECT * FROM stale
)
SELECT
  u.* EXCEPT (openalex_created_dt),
  u.openalex_created_dt,
  CASE
    WHEN h.payload_hash = xxhash64(to_json(struct(
      u.work_id, u.work_id_source, u.merge_key, u.key_lineage, u.title, u.normalized_title,
      u.authors, u.ids, u.raw_type, u.type, u.version, u.license, u.language, u.language_classification,
      u.published_date, u.created_date, u.updated_date, u.issue, u.volume, u.first_page, u.last_page,
      u.is_retracted, u.abstract, u.source_name, u.publisher, u.funders, u.references, u.urls,
      u.pdf_url, u.landing_page_url, u.pdf_s3_id, u.grobid_s3_id, u.mesh, u.is_oa, u.is_oa_source,
      u.referenced_works_count, u.referenced_works, u.abstract_inverted_index,
      u.authors_exist, u.affiliations_exist, u.is_corresponding_exists, u.best_doi, u.source_id
    ))) THEN h.openalex_updated_dt
    ELSE current_timestamp()
  END AS openalex_updated_dt
FROM unioned u
LEFT JOIN identifier('openalex' || :env_suffix || '.works.locations_mapped_hash') h
  ON  u.provenance = h.provenance
  AND u.native_id_namespace = h.native_id_namespace
  AND u.native_id = h.native_id
  AND u.work_id <=> h.work_id

## Refresh the hash table from the rebuilt output

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.locations_mapped_hash')
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
SELECT provenance, native_id_namespace, native_id, work_id,
       xxhash64(to_json(struct(
         work_id, work_id_source, merge_key, key_lineage, title, normalized_title,
         authors, ids, raw_type, type, version, license, language, language_classification,
         published_date, created_date, updated_date, issue, volume, first_page, last_page,
         is_retracted, abstract, source_name, publisher, funders, references, urls,
         pdf_url, landing_page_url, pdf_s3_id, grobid_s3_id, mesh, is_oa, is_oa_source,
         referenced_works_count, referenced_works, abstract_inverted_index,
         authors_exist, affiliations_exist, is_corresponding_exists, best_doi, source_id
       ))) AS payload_hash,
       openalex_updated_dt
FROM identifier('openalex' || :env_suffix || '.works.locations_mapped')
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY provenance, native_id_namespace, native_id, work_id
  ORDER BY openalex_updated_dt DESC NULLS LAST) = 1

In [0]:
SELECT format_number(COUNT(*), 0) AS row_count,
       format_number(COUNT(DISTINCT work_id), 0) AS distinct_works
FROM identifier('openalex' || :env_suffix || '.works.locations_mapped')